In [12]:
from typing import List, Optional
import requests
def get_embeddings(sentences: List[str], max_length: Optional[int] = 256, url: str = "http://192.168.18.237:8025/embeddings_normalize_query") -> List[List[float]]:
    """
    获取句子的嵌入向量。

    :param sentences: 需要编码的句子列表。
    :param max_length: 句子的最大长度（可选，默认为 256）。
    :param url: FastAPI 接口的 URL（可选，默认为本地服务地址）。
    :return: 返回句子的嵌入向量列表。
    """
    # 构造请求体
    data = {
        "content": sentences,
        "max_length": max_length
    }

    try:
        # 发送 POST 请求
        response = requests.post(url, json=data)
        response.raise_for_status()  # 检查请求是否成功
        result = response.json()  # 解析 JSON 响应
        return result["embeddings"]
    except requests.exceptions.RequestException as e:
        print(f"请求失败: {e}")
        return []
    except KeyError:
        print("响应格式错误，未找到 'embeddings' 字段。")
        return []

In [2]:
import os
import pandas as pd
import pickle
from tqdm import tqdm  # 导入 tqdm 库

# 加载现有的 embedding_dict，如果没有则初始化一个空的字典
embedding_dict_file = '/Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/embedding_dict_STOCK_SNAME_updated.pkl'
if os.path.exists(embedding_dict_file):
    with open(embedding_dict_file, 'rb') as f:
        embedding_dict = pickle.load(f)
        print(f"已加载现有的嵌套字典 {embedding_dict_file}")
else:
    embedding_dict = {}
    print("未检测到现有的嵌套字典，已初始化新的嵌套字典")

# 定义要处理的表和字段，以及对应的文件路径
table_field_mapping = {
    'VIEW_COM3105': {
        'field_name': 'F006v',  # 产品名字段
        'company_field': 'CSNAME'  # 公司名字段
    },
    'VIEW_COM3106': {
        'field_name': 'F006v',  # 产品名字段
        'company_field': 'CSNAME'  # 公司名字段
    }
}

# 设置数据文件夹路径
data_folder = '/Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US'  # 请根据您的实际情况修改

# 遍历指定的表和字段
for table_name, mapping in tqdm(table_field_mapping.items(), desc="处理表"):
    field_name = mapping['field_name']
    company_field = mapping['company_field']

    # 构造 CSV 文件路径
    csv_file = os.path.join(data_folder, f"{table_name}.csv")

    # 检查文件是否存在
    if not os.path.exists(csv_file):
        print(f"文件 {csv_file} 不存在，跳过。")
        continue

    # 读取 CSV 文件
    df = pd.read_csv(csv_file)

    # 检查字段是否存在
    required_fields = [field_name, company_field]
    for field in required_fields:
        if field not in df.columns:
            print(f"表 {table_name} 中不存在字段 {field}，跳过。")
            continue

    # 获取公司名和字段值的组合
    unique_values = df[[company_field, field_name]].dropna().values.tolist()

    # 去重
    unique_values = list(set([tuple(row) for row in unique_values]))

    # 构建字段值与嵌入向量的映射
    if table_name not in embedding_dict:
        embedding_dict[table_name] = {}

    if field_name not in embedding_dict[table_name]:
        embedding_dict[table_name][field_name] = {}

    # 使用 tqdm 显示处理公司-产品对的进度
    for company, product in tqdm(unique_values, desc=f"处理 {table_name}.{field_name}"):
        if company not in embedding_dict[table_name][field_name]:
            embedding_dict[table_name][field_name][company] = {}

        # 假设 get_embeddings 是一个函数，返回产品的嵌入向量
        embedding = get_embeddings([product])[0]  # 获取单个产品的嵌入向量
        embedding_dict[table_name][field_name][company][product] = embedding

    print(f"已处理表 {table_name} 的字段 {field_name}，包含 {len(unique_values)} 个字段值。")

# 将更新后的嵌套字典保存为新的 .pkl 文件
new_embedding_dict_file = '/Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/VIEW_america_embedding_dict_updated.pkl'  # 您可以修改为您想要的文件名或路径
with open(new_embedding_dict_file, 'wb') as f:
    pickle.dump(embedding_dict, f)

print(f"嵌套字典已更新并保存到新文件 {new_embedding_dict_file}")

已加载现有的嵌套字典 /Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/embedding_dict_STOCK_SNAME_updated.pkl


处理表:   0%|          | 0/2 [01:35<?, ?it/s]


KeyboardInterrupt: 

In [13]:
import os
import pandas as pd
import pickle
from tqdm import tqdm

# 加载现有的 embedding_dict，如果没有则初始化一个空的字典
embedding_dict_file = '/Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/embedding_dict_STOCK_SNAME_updated.pkl'
if os.path.exists(embedding_dict_file):
    with open(embedding_dict_file, 'rb') as f:
        embedding_dict = pickle.load(f)
        print(f"已加载现有的嵌套字典 {embedding_dict_file}")
else:
    embedding_dict = {}
    print("未检测到现有的嵌套字典，已初始化新的嵌套字典")

# 定义要处理的表和字段，以及对应的文件路径
table_field_mapping = {
    'VIEW_COM3105': {
        'field_name': 'F006v',  # 产品名字段
        'company_field': 'CSNAME'  # 公司名字段
    },
    'VIEW_COM3106': {
        'field_name': 'F006v',  # 产品名字段
        'company_field': 'CSNAME'  # 公司名字段
    }
}

# 设置数据文件夹路径
data_folder = '/Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US'

# 设置批次大小
batch_size = 1000  # 你可以根据需要调整这个值

# 遍历指定的表和字段
for table_name, mapping in tqdm(table_field_mapping.items(), desc="处理表"):
    field_name = mapping['field_name']
    company_field = mapping['company_field']

    # 构造 CSV 文件路径
    csv_file = os.path.join(data_folder, f"{table_name}.csv")

    # 检查文件是否存在
    if not os.path.exists(csv_file):
        print(f"文件 {csv_file} 不存在，跳过。")
        continue

    # 读取 CSV 文件
    df = pd.read_csv(csv_file)

    # 检查字段是否存在
    required_fields = [field_name, company_field]
    for field in required_fields:
        if field not in df.columns:
            print(f"表 {table_name} 中不存在字段 {field}，跳过。")
            break
    else:  # 只有当没有 break 时才执行，Python 的 for-else 结构
        # 获取公司名和字段值的组合
        unique_values = df[[company_field, field_name]].dropna().values.tolist()

        # 去重
        unique_values = list(set([tuple(row) for row in unique_values]))

        # 初始化嵌套字典结构（如果尚不存在）
        if table_name not in embedding_dict:
            embedding_dict[table_name] = {}
        if field_name not in embedding_dict[table_name]:
            embedding_dict[table_name][field_name] = {}

        # 分批次处理数据
        for i in tqdm(range(0, len(unique_values), batch_size), desc=f"分批次处理 {table_name}.{field_name}"):
            batch = unique_values[i:i + batch_size]
            products_to_embed = [product for _, product in batch]

            # 批量获取嵌入向量
            embeddings = get_embeddings(products_to_embed)

            # 将嵌入向量添加到字典中
            for (company, product), embedding in zip(batch, embeddings):
                if company not in embedding_dict[table_name][field_name]:
                    embedding_dict[table_name][field_name][company] = {}
                embedding_dict[table_name][field_name][company][product] = embedding

        print(f"已处理表 {table_name} 的字段 {field_name}，包含 {len(unique_values)} 个字段值。")

# 将更新后的嵌套字典保存为新的 .pkl 文件
new_embedding_dict_file = '/Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/1VIEW_america_embedding_dict_updated.pkl'
with open(new_embedding_dict_file, 'wb') as f:
    pickle.dump(embedding_dict, f)

print(f"嵌套字典已更新并保存到新文件 {new_embedding_dict_file}")

已加载现有的嵌套字典 /Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/embedding_dict_STOCK_SNAME_updated.pkl


处理表:  50%|█████     | 1/2 [10:10<10:10, 610.78s/it]

已处理表 VIEW_COM3105 的字段 F006v，包含 28180 个字段值。



处理表: 100%|██████████| 2/2 [10:27<00:00, 314.00s/it]


已处理表 VIEW_COM3106 的字段 F006v，包含 1099 个字段值。
嵌套字典已更新并保存到新文件 /Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/1VIEW_america_embedding_dict_updated.pkl


In [14]:
import os
import pickle
embedding_dict_file = '/Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/1VIEW_america_embedding_dict_updated.pkl'
if os.path.exists(embedding_dict_file):
    with open(embedding_dict_file, 'rb') as f:
        embedding_dict = pickle.load(f)
        print(f"已加载现有的嵌套字典 {embedding_dict_file}")
else:
    embedding_dict = {}
    print("未检测到现有的嵌套字典，已初始化新的嵌套字典")

已加载现有的嵌套字典 /Users/rabyte_c/Desktop/experimental/text2sql_renew/vector-US/1VIEW_america_embedding_dict_updated.pkl


In [2]:
db_type = 'MYSQL-1'
username, password, dsn, host, port = "", "", "", "", None
if db_type == "MYSQL-1":
    username ='etl'
    password = '5RDsS56g13UU^&O&v'
    host = 'mr-bth0s2yfzy228yweuw.rwlb.rds.aliyuncs.com'
    port = 3306
elif db_type == "MYSQL-2":
    username ='juling'
    password = 'G&N$Ls9LVVMBgL49'
    host = 'rm-2ze3bb361946ye7oq.mysql.rds.aliyuncs.com'
    port = 3306
elif db_type == "ORCL":
    username ='ngdp_readonly'
    password = 'eBi#xDPKGTpGAC'
    host = 'prd-rabyte-ngdp.cmz9yspwe5fc.rds.cn-northwest-1.amazonaws.com.cn'
    port = 1521
    service_name = 'ORCL'
else:
    assert db_type in ["MYSQL-1", "MYSQL-2"], f"{db_type}为未支持数据库类型"


import aioodbc
import pandas as pd
from typing import Tuple, Any
#在代码前添加驱动检查
import pyodbc
print("Available ODBC Drivers:", pyodbc.drivers())  # 确认能看到Oracle驱动
async def fetch_data_from_ORACLE(
    final_sql: str,
    host: str,
    user: str,
    password: str,
    port: int = 1521,
    service_name: str = "ORCL",
) -> Tuple[int, Any]:
    """
    修正后的Oracle异步连接版本
    """
    status_code = 200
    return_res = None

    try:
        # 更可靠的连接字符串格式
        dsn = f"""
            DRIVER={{Oracle ODBC Driver}};
            SERVER={host};
            PORT={port};
            SERVICE_NAME={service_name};
            UID={user};
            PWD={password}
        """.replace("\n", "").strip()

        # 添加连接超时参数
        conn_params = {
            "dsn": dsn,
            "timeout": 30  # 增加超时设置
        }

        # 测试连接
        print(f"Attempting to connect with DSN: {dsn}")
        connection = await aioodbc.connect(**conn_params)

        async with connection.cursor() as cursor:
            await cursor.execute(final_sql)
            rows = await cursor.fetchmany(30000)

            if rows:
                cols = [desc[0] for desc in cursor.description]
                return_res = pd.DataFrame(rows, columns=cols)
            else:
                return_res = "No data found"
                status_code = 404

    except aioodbc.Error as e:
        status_code = 500
        return_res = f"ODBC Error: {str(e)}"
        print(f"ODBC Error detail: {e}")

    except Exception as e:
        status_code = 500
        return_res = f"System Error: {str(e)}"
        # print(f"Error traceback: {traceback.format_exc()}")

    return status_code, return_res


ImportError: dlopen(/Users/rabyte_c/WorkSpace/SoftWare/miniconda3/envs/edb_ebv/lib/python3.9/site-packages/pyodbc.cpython-39-darwin.so, 0x0002): Library not loaded: /opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib
  Referenced from: <6B124F4D-6A35-32F2-89C5-8FFAADF7D234> /Users/rabyte_c/WorkSpace/SoftWare/miniconda3/envs/edb_ebv/lib/python3.9/site-packages/pyodbc.cpython-39-darwin.so
  Reason: tried: '/opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib' (no such file), '/opt/homebrew/opt/unixodbc/lib/libodbc.2.dylib' (no such file)